# 🔧 04 — Orthogonalize Bobot & Save Model

**Tujuan:** Terapkan orthogonalization ke bobot model secara permanen — ini inti dari abliteration.

**Lokasi:** VS Code (ringan, cuma operasi matriks sekali jalan)

---
**Reference:** ortho_cookbook.ipynb (FailSpy), mlabonne abliteration article

**Improvement:** Support `REFUSAL_WEIGHT` parameter — kontrol seberapa kuat ablasi.

**Output:** `models/abliterated/ablation.safetensors`

Saved results are in `results/reference/`. Your results may differ with other inputs or settings. See `docs/results.md` for details.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from inside the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import einops
from tqdm.notebook import tqdm
from safetensors.torch import load_file, save_file
from transformer_lens import HookedTransformer

from src.config import DEFAULT_MODEL, DEVICE, ABLITERATED_DIR, SEED, REFUSAL_WEIGHT

torch.manual_seed(SEED)
torch.set_grad_enabled(False)  # hemat GPU memory (credit: Undi95)
print(f"Device: {DEVICE}")
print(f"REFUSAL_WEIGHT: {REFUSAL_WEIGHT} (>1 = stronger ablation)")

In [ ]:
# ─── Load model ───
model = HookedTransformer.from_pretrained_no_processing(
    DEFAULT_MODEL,
    dtype=torch.bfloat16,
    default_padding_side='left',
    device=DEVICE,
)
model.tokenizer.padding_side = 'left'
model.tokenizer.pad_token = model.tokenizer.eos_token
print(f"Model loaded: {model.cfg.model_name}")
print(f"n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}")

In [ ]:
# --- Load refusal direction ---
refusal_dirs = load_file(str(ABLITERATED_DIR / 'refusal_dirs.safetensors'))

# --- Baca hasil screening untuk pilih layer terbaik ---
import pandas as pd
csv_path = ABLITERATED_DIR / 'screening_results.csv'
if csv_path.exists():
    df_scr = pd.read_csv(csv_path)
    best = df_scr.loc[df_scr['refusal_rate'].idxmin()]
    dir_key = f"resid_pre.layer_{int(best['layer'])}"
    print(f"Auto-picked: {dir_key} (RR = {best['refusal_rate']:.2%})")
else:
    dir_key = 'resid_pre.layer_7'  # fallback
    print(f"Screening CSV not found, using fallback: {dir_key}")

refusal_dir = refusal_dirs[dir_key]
print(f"Shape: {refusal_dir.shape}, Norm: {refusal_dir.norm():.4f}")
print(f'REFUSAL_WEIGHT: {REFUSAL_WEIGHT}')


### Orthogonalize Weights

Ada 3 komponen yang di-orthogonalize per layer:
1. **W_E** (embedding) — 1x
2. **W_O** (attention output) — per layer
3. **W_out** (MLP output) — per layer

Formula: `W' = W - weight * (W · r̂) r̂`

Reference: ortho_cookbook (FailSpy), mlabonne

In [ ]:
def get_orthogonalized_matrix(matrix, vec, weight=REFUSAL_WEIGHT):
    """
    Orthogonalize matrix terhadap direction vector.
    weight: kontrol seberapa kuat ablasi (1.0 = standar, >1 = lebih kuat)
    """
    vec = vec.view(-1).to(matrix.device)
    proj = einops.einsum(
        matrix, vec.view(-1, 1),
        '... d_model, d_model single -> ... single'
    ) * vec
    return matrix - weight * proj


# ─── Apply orthogonalization ───
print("Orthogonalizing W_E (embedding)...")
model.W_E.data = get_orthogonalized_matrix(model.W_E.data, refusal_dir)

for i, block in enumerate(tqdm(model.blocks, desc="Orthogonalizing layers")):
    block.attn.W_O.data = get_orthogonalized_matrix(block.attn.W_O.data, refusal_dir)
    block.mlp.W_out.data = get_orthogonalized_matrix(block.mlp.W_out.data, refusal_dir)

print("Orthogonalization complete!")

In [ ]:
# ─── Simpan state dict yang sudah di-orthogonalize ───
state_dict = model.state_dict()
save_file(state_dict, str(ABLITERATED_DIR / 'ablation.safetensors'))
print(f"Saved: {ABLITERATED_DIR / 'ablation.safetensors'}")
print(f"Keys: {len(state_dict)}")
print(f"Size: {sum(v.numel() * v.element_size() for v in state_dict.values()) / 1e6:.1f} MB")

# Simpan juga refusal_dir yang dipilih untuk dokumentasi
save_file({dir_key: refusal_dir.cpu()}, str(ABLITERATED_DIR / 'selected_refusal_dir.safetensors'))
print(f"\nSaved selected direction: {dir_key}")

---
**Next:** Lanjut ke `05_convert_to_hf.ipynb` untuk konversi ke format HuggingFace.